### formular una señal como una hipótesis falsable y distinguir predictibilidad de rentabilidad.

<b> Probabilidad/estadística </b> | ¿Tu señal realmente contiene información? Un researcher afirma: “cuando el retorno de los últimos 5 días es positivo, el siguiente día tiene mayor probabilidad de ser positivo”. 

En 1,000 observaciones encuentras 560 casos con momentum positivo; de éstos, 308 tuvieron retorno positivo al día siguiente. Calcula $(\hat p=308/560)$, plantea $(H_0:p=0.5)$ contra $(H_1:p>0.5)$ y calcula

$$ z=\frac{\hat p-0.5}{\sqrt{0.5(1-0.5)/560}}. $$

Decide aproximadamente si rechazarías $(H_0)$ al 5%. Después responde algo más importante que el p-value: aunque $(p>0.5)$ fuese estadísticamente convincente.

¿qué información todavía te falta para afirmar que existe una oportunidad de trading?
- Falta demostrar que la señal se traduce a PnL neto y no es un hallazgo accidental 
  - Magnitud economica: ver el retorno promedio/mediano condicionado a la señal y su distribucion. un 50.1%de aciertos puede perder dinero si las perdidas son mayores a las gancias 
  - costos de implementación: comisiones, spread, slippage 
  - regla operable : definir exactamente entrada, salida, tamaño de posicion, horarios, universo de activos, evitar look-ahead bias 
  - Riesgo : volatilidad, dradown, sharpe/sortino, exposicion a factores, comportamineto a crisis 
  - robustez : validacion fuera de muestra, walk-forward
  - data snooping: si se probo muchas señales, horizontes o umbrales, el p-value aislado deja de ser suficiente
  -estabilidad : verifica que el efecto no dependa de unos pocos dias, activos o regimenes de mercado

In [4]:
import numpy as np 
import scipy.stats as stats
p = 308/560
z_score = (p-0.5)/ (np.sqrt(0.5*(0.5) / 560))
p_value = stats.norm.sf(z_score) # sf : 1 - cdf ; mas preciso numericamente 
alfa = .05
if p_value < alfa: 
    print('Rechazamos H0')
else : 
    print('No rechazamos H0; No hay evidencia suficiente para rechazarlo')



Rechazamos H0


#### Código | Accuracy no es PnL

Simular 2,000 trades donde tu modelo acierta la dirección con probabilidad 0.56. 

Cuando acierta, genera +8 bps; cuando falla, -12 bps. Calcula accuracy, mean_return_bps, std_return_bps y PnL acumulado.

Después repite cambiando solamente el payoff a +12/-8 bps. Mantén aproximadamente la misma accuracy. 

Tu entregable debe ser una tabla pequeña comparando ambos experimentos. Explica en dos líneas por qué un clasificador con 56% de accuracy puede ser una estrategia mala y otro con exactamente la misma accuracy puede ser útil. Ésta es una conexión muy importante entre ML y trading que conviene tener automatizada mentalmente.

In [8]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

n_trades = 2_000
p_acierto = 0.56

# True = acierto; False = fallo
aciertos = rng.random(n_trades) < p_acierto

# Retorno por trade, en bps
retornos_bps = np.where(aciertos, 8, -12)

trades = pd.DataFrame({
    "acierto": aciertos,
    "retorno_bps": retornos_bps,
})

trades["pnl_acumulado_bps"] = trades["retorno_bps"].cumsum()
accuracy = trades["acierto"].mean()
mean_return_bps = trades["retorno_bps"].mean()
std_return_bps = trades["retorno_bps"].std()  # desviación estándar muestral

print(f"Accuracy: {accuracy:.2%}")
print(f"Mean return: {mean_return_bps:.2f} bps")
print(f"Std return: {std_return_bps:.2f} bps")

Accuracy: 55.85%
Mean return: -0.83 bps
Std return: 9.93 bps


In [6]:
print("Tasa de acierto:", trades["acierto"].mean())
print("PnL total:", trades["retorno_bps"].sum(), "bps")
print("PnL promedio por trade:", trades["retorno_bps"].mean(), "bps")

Tasa de acierto: 0.5585
PnL total: -1660 bps
PnL promedio por trade: -0.83 bps


In [7]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(seed=42)

n_trades = 2_000
p_acierto = 0.56

# True = acierto; False = fallo
aciertos = rng.random(n_trades) < p_acierto

# Retorno por trade, en bps
retornos_bps = np.where(aciertos, 12, -8)

trades = pd.DataFrame({
    "acierto": aciertos,
    "retorno_bps": retornos_bps,
})

trades["pnl_acumulado_bps"] = trades["retorno_bps"].cumsum()

trades.head()
print("Tasa de acierto:", trades["acierto"].mean())
print("PnL total:", trades["retorno_bps"].sum(), "bps")
print("PnL promedio por trade:", trades["retorno_bps"].mean(), "bps")

Tasa de acierto: 0.5585
PnL total: 6340 bps
PnL promedio por trade: 3.17 bps


La idea central es:
- Trading value $\neq$ Accuracy
- Trading value $ \approx$ expected return neto de costos, ajustado por riesgo

Un clasificador con 51% de accuracy puede ser excelente si selecciona bien los pocos movimientos grandes y limita las pérdidas. Y uno con 70% puede ser desastroso si cada error ocurre en un movimiento adverso muy grande.

---
#### cómo saber si un resultado de research probablemente generaliza o si simplemente encontraste algo por buscar demasiado.

Multiple testing y falsos descubrimientos. Imagina que pruebas 20 señales completamente inútiles y para cada una haces un hypothesis test con $(\alpha=5\%)$. Bajo $(H_0)$, calcula el número esperado de falsos positivos, $(E[FP]=m\alpha)$, y después la probabilidad de encontrar al menos un resultado significativo:

$ P(FP\ge1)=1-(1-\alpha)^m. $


Repite para \(m=100\). Después contesta: si probaste 100 estrategias y una obtiene $(p=0.03)$, ¿por qué ya no puedes interpretarla igual que si esa hubiese sido tu única hipótesis? Finalmente calcula el threshold de Bonferroni $(\alpha=0.05/m)$ para ambos casos.

In [9]:
import numpy as np
import pandas as pd

alpha = 0.05

resultados = []

for m in [20, 100]:
    falsos_positivos_esperados = m * alpha
    prob_al_menos_uno = 1 - (1 - alpha)**m
    bonferroni = alpha / m

    resultados.append({
        "señales_probadas": m,
        "E[FP]": falsos_positivos_esperados,
        "P(al menos 1 FP)": prob_al_menos_uno,
        "threshold_Bonferroni": bonferroni
    })

tabla = pd.DataFrame(resultados)
tabla

,señales_probadas,E[FP],P(al menos 1 FP),threshold_Bonferroni
0,20,1.0,0.641514,0.0025
1,100,5.0,0.994079,0.0005


<b> probabilidad | Misma volatilidad:</b> distinta cola. Genera dos series de 10_000 retornos:

- Comprueba que ambas tienen aproximadamente la misma volatilidad. Para cada distribución calcula:

$ 1. \ P(R<−2\%) \\ 2. VaR_{99\%} \\ 3. ES_{99\%} $

--- 
si solamente hubieras pasado volatility = 1% al Risk Engine, ¿qué diferencia entre estos dos activos habría quedado completamente escondida?

- Si el motor de riesgo solo recibiera una volatilidad de 1%, trataría ambos activos como igual de riesgosos. Quedaría escondido que el activo con distribución Student-\(t\) tiene colas más pesadas: aunque su variación típica es igual, presenta mayor probabilidad de pérdidas extremas y pérdidas promedio más severas dentro del peor 1% de escenarios. Por ello, la volatilidad por sí sola subestimaría su riesgo de cola.

In [ ]:
import numpy as np 
import scipy.stats as stats 
import  pandas as pd 
# 1 generamos 2 series de retornos de 10,000 observaciones
#generador reproducible
rng = np.random.default_rng(seed= 42)
# serie A: normal 
# serie B: t-student con pocos grados de libertad, reescalada para que la desviacion sea 1% 
serie_a = rng.normal(0, 0.01, 10_000)

t = rng.standard_t(df=3, size=10_000)
serie_b = t / t.std(ddof=1) * 0.01

# comprobamos que ambas series tengan una volatilidad similar 
vol_a  = np.std(serie_a)
vol_b  = np.std(serie_b)
print(f'Volatilidad serie A: {vol_a:.2f}')
print(f'Volatilidad serie B: {vol_b:.2f}')
# para cada distribucion calculamos P(R < -2%), Var99%, ES99% 
def metricas_riesgo(retornos): 
    var_99 = np.quantile(retornos,0.01)
    es_99 = retornos[retornos <= var_99].mean()
    return {'volatilidad' : retornos.std(ddof=1),
            'P(R < -2%)': (retornos < -0.02).mean(), 
            'VaR_99%': var_99, 
            'ES_99%' : es_99}
# las 2 series se guardan en un diccionario y la transformabamos
series = {'normal' : serie_a, 
          't-student': serie_b}
resultados = []
for nombre, retornos in series.items():
    fila ={'distribucion':nombre}
    fila.update(metricas_riesgo(retornos))
    resultados.append(fila)
df_metricas = pd.DataFrame(resultados)
df_metricas

Volatilidad serie A: 0.01
Volatilidad serie B: 0.01


,distribucion,volatilidad,P(R < -2%),VaR_99%,ES_99%
0,normal,0.009951,0.0230,-0.023229,-0.02642
1,t-student,0.010001,0.0206,-0.026153,-0.03981


<b> Código + Quant Desk | </b> De risk metric a risk limit.

Tu portfolio vale USD 100,000. Supón que de tus escenarios históricos/simulados obtienes:

$$ VaR_{99}= \$2,400,\qquad ES_{99}= \$4,100. $$

Tu política establece:

$$ ES_{99}\le 3\%\times NAV. $$

Primero determina si el portfolio viola el límite.

Después supone, sólo para este ejercicio, que las pérdidas escalan linealmente con exposición. Calcula el factor

$ k = \frac{ES_{current}}{​ES_{limit}}​ $ 

y úsalo para reducir proporcionalmente unos pesos actuales:

w=(0.40,0.35,0.25).

Obtén $(w^{*}=kw)$ y cuánto queda en cash.